# 🛡️ UYDS SSS Cevap Doğrulama Pipeline v4.1.0
## Qwen3-32B-AWQ · vLLM Structured Output · Tokenizer Chunking · Modüler Mimari

**v4.1.0 yenilikler:**
- 🆕 **Modüler mimari:** 12 `src/` modülü, notebook sadece ince yürütme katmanı
- 🆕 **Tokenizer tabanlı chunking:** Embedding tokenizer ile gerçek token sayımı (110/25)
- 🆕 **Kaynak envanteri:** SHA-256 hash, kopya tespiti, tarih/sürüm çıkarma
- 🆕 **Çok formatlı belge okuyucu:** PDF/DOCX/TXT/MD/XLSX/CSV/JSON/JSONL
- 🆕 **Smoke test:** Mock model ile 20 kayıt uçtan uca test
- 🆕 **71 birim test:** excel_format, chunking, retrieval, prompt, json_validation, checkpoint, output_integrity
- 🔧 vLLM 0.12.0 · CUDA 12.8 · StructuredOutputsParams
---

## ⚙️ Hücre 0: Kurulum + Paketler + GPU Kontrolü

Google Drive'ı bağlar, tüm paketleri kurar, GPU'yu doğrular.

In [ ]:
# ============================================================
# HÜCRE 0: Kurulum + Paketler + GPU Kontrolü
# ============================================================
import os, sys, gc, time

# --- Google Drive bağlama ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIZINI = "/content/drive/MyDrive"
    print("🖥️ Colab — Drive bağlandı")
except:
    DRIVE_DIZINI = "/content"
    print("💻 Lokal — Drive yok")

# --- GPU kontrolü ---
import torch
if not torch.cuda.is_available():
    raise RuntimeError("❌ CUDA GPU bulunamadı! A100 40GB+ seçin.")
gpu_adi = torch.cuda.get_device_name(0)
gpu_bellek = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"✅ GPU: {gpu_adi}, {gpu_bellek:.0f} GB")
if gpu_bellek < 38:
    raise RuntimeError(f"GPU belleği yetersiz: {gpu_bellek:.1f} GB")

# --- Paket kurulumları ---
!pip install -qU pip setuptools wheel 2>&1 | tail -1
!pip uninstall -y vllm 2>&1 | tail -1
!pip install -qU vllm==0.12.0 --extra-index-url https://download.pytorch.org/whl/cu128 2>&1 | tail -3
!pip install -qU pyarrow==18.1.0 2>&1 | tail -1
!pip install -qU transformers sentence-transformers pypdf python-docx openpyxl matplotlib pyyaml 2>&1 | tail -1

print("✅ Hücre 0 tamam.")

## 📁 Hücre 1: Proje Yapılandırması + Config Yükleme

`src/config.py` modülü tüm yolları ve eşikleri yönetir.

In [ ]:
# ============================================================
# HÜCRE 1: Config + Dizinler
# ============================================================
import sys
PROJE_DIZINI = os.path.join(DRIVE_DIZINI, "google_collab_egitim")
if not os.path.isdir(PROJE_DIZINI):
    raise FileNotFoundError(f"Proje dizini bulunamadı: {PROJE_DIZINI}")
sys.path.insert(0, PROJE_DIZINI)

from src.config import (
    KAYNAK_DIZINI, INPUT_DIZINI, CIKTI_DIZINI, CKPT_DIZINI,
    EMBEDDING_MODEL, EMBEDDING_BATCH, SORU_EMB_BATCH,
    BIRINCIL_MODEL, YEDEK_MODEL,
    CHUNK_BOYUT_TOKEN, CHUNK_OVERLAP_TOKEN, CHUNK_ALGORITMA_SURUMU,
    RAG_TOP_K, RAG_SON_K, RAG_MIN_SKOR,
    GUVEN_ESIK, MAX_MODEL_LEN, GPU_MEM_UTIL, BATCH_BUYUKLUK,
    MAX_NEW_TOKENS, PROMPT_TOKEN_LIMIT,
    PROMPT_SURUMU, KURAL_SETI_SURUMU, CONFIG_HASH,
    dizin_hashi, dosya_hashi, dizinleri_hazirla, config_ozeti,
)

dizinleri_hazirla()
print(config_ozeti())
print(f"\n📁 Proje: {PROJE_DIZINI}")
print(f"📂 Çıktı: {CIKTI_DIZINI}")
print(f"📂 Checkpoint: {CKPT_DIZINI}")
print("✅ Hücre 1 tamam.")

## 📖 Hücre 2: Kaynak Havuzu — Belge Okuma + Chunking + Embedding

`src/document_loader`, `src/chunking`, `src/source_inventory`, `src/embeddings` kullanılır.

In [ ]:
# ============================================================
# HÜCRE 2: Kaynak Havuzu İnşası
# ============================================================
from src.document_loader import tum_kaynaklari_yukle
from src.chunking import belgeyi_chunkla, chunk_istatistikleri
from src.source_inventory import envanter_olustur, envanter_ozeti
from src.embeddings import (
    embedding_modeli_yukle, kaynak_embedding_olustur,
    embedding_checkpoint_kaydet, embedding_checkpoint_yukle,
)
from src.utils import Zamanlayici

z = Zamanlayici()

# 1. Envanter
envanter = envanter_olustur(
    KAYNAK_DIZINI,
    cikti_yolu=os.path.join(KAYNAK_DIZINI, "kaynak_envanteri.csv"),
)
ozet = envanter_ozeti(envanter)
print(f"📋 {ozet['benzersiz_kaynak']} benzersiz kaynak")
z.kontrol("envanter")

# 2. Belge okuma
belgeler = tum_kaynaklari_yukle(KAYNAK_DIZINI)
print(f"📄 {len(belgeler)} sayfa/belge okundu")
z.kontrol("belge_okuma")

# 3. Tokenizer tabanlı chunking
try:
    from transformers import AutoTokenizer
    tk = AutoTokenizer.from_pretrained(EMBEDDING_MODEL, trust_remote_code=False)
    print(f"🔤 Tokenizer: {EMBEDDING_MODEL}")
except:
    tk = None
    print("⚠️ Tokenizer yok, kelime bazlı fallback")

tum_parcalar = belgeyi_chunkla(belgeler, tk, hedef_token=CHUNK_BOYUT_TOKEN, overlap_token=CHUNK_OVERLAP_TOKEN)
ch_stats = chunk_istatistikleri(tum_parcalar, tk)
print(f"✂️ {ch_stats['toplam_chunk']} chunk, ortalama {ch_stats['ortalama_token']:.0f} token")
z.kontrol("chunking")

# 4. Embedding (checkpoint'ten yükle veya sıfırdan oluştur)
KAYNAK_HASH = dizin_hashi(KAYNAK_DIZINI)
emb_durum = embedding_checkpoint_yukle(
    CKPT_DIZINI, KAYNAK_HASH, EMBEDDING_MODEL,
    CHUNK_BOYUT_TOKEN, CHUNK_OVERLAP_TOKEN, CHUNK_ALGORITMA_SURUMU,
)

if emb_durum:
    kaynak_vektorler, tum_parcalar = emb_durum
else:
    emb_model = embedding_modeli_yukle(EMBEDDING_MODEL, cihaz="cuda")
    metinler = [p["metin"] for p in tum_parcalar]
    kaynak_vektorler = kaynak_embedding_olustur(metinler, emb_model, batch_size=EMBEDDING_BATCH)
    embedding_checkpoint_kaydet(
        CKPT_DIZINI, kaynak_vektorler, tum_parcalar,
        KAYNAK_HASH, EMBEDDING_MODEL,
        CHUNK_BOYUT_TOKEN, CHUNK_OVERLAP_TOKEN, CHUNK_ALGORITMA_SURUMU,
    )
    del emb_model; gc.collect()

print(f"🔤 {kaynak_vektorler.shape[0]} vektör × {kaynak_vektorler.shape[1]}d")
z.kontrol("embedding")
print(f"\n⏱️ Kaynak havuzu: {z.gecen():.1f}s")
print("✅ Hücre 2 tamam.")

## 📊 Hücre 3: Excel Okuma + RAG Retrieval

UYDS Excel'ini okur, soru/cevap alanlarını eşleştirir, her soru için en uygun kaynakları getirir.

In [ ]:
# ============================================================
# HÜCRE 3: Excel Okuma + RAG
# ============================================================
import glob, unicodedata
import numpy as np
import pandas as pd
from src.utils import adi_normalize_et
from src.embeddings import soru_embedding_olustur
from src.retrieval import toplu_kaynak_ara, rag_istatistikleri

# --- Excel bul ve oku ---
excel_dosyalari = glob.glob(os.path.join(INPUT_DIZINI, "*.xlsx"))
if not excel_dosyalari:
    raise RuntimeError(f"❌ {INPUT_DIZINI} içinde .xlsx bulunamadı!")
EXCEL_DOSYASI = excel_dosyalari[0]
print(f"📊 Excel: {os.path.basename(EXCEL_DOSYASI)}")

# Başlık satırı algılama (v4.0.2'den)
ham = pd.read_excel(EXCEL_DOSYASI, sheet_name=0, header=None, engine="openpyxl", dtype=object)
bilinen = {"modul", "musteri", "konu", "aciklama", "yapilacak", "yapilan islem", "kategori"}
baslik_satiri = next((i for i, satir in ham.head(20).iterrows()
                      if len({adi_normalize_et(str(d)) for d in satir.tolist()} & bilinen) >= 3), None)
if baslik_satiri is None:
    raise ValueError("Excel başlık satırı bulunamadı.")
veri = pd.read_excel(EXCEL_DOSYASI, sheet_name=0, header=baslik_satiri, engine="openpyxl", dtype=object)
veri = veri.dropna(axis=1, how="all")
print(f"✅ Başlık: {baslik_satiri + 1}. satır, {len(veri)} kayıt")

# Alan eşleme
mk = {adi_normalize_et(k): k for k in veri.columns}
H = {"modul": ["modul"], "konu": ["konu"], "aciklama": ["aciklama"], "yapilacak": ["yapilacak"],
     "yapilan_islem": ["yapilan islem", "yapilan_islem"]}

def kle(h): return {a: mk[adi_normalize_et(ad)] for a, ads in h.items() for ad in ads if adi_normalize_et(ad) in mk}
eslesme = kle(H)
ft = "uyds" if len(eslesme) >= 4 else ("standart" if len(kle({"soru": ["soru"], "cevap": ["cevap"]})) >= 2 else None)
print(f"✅ Format: {ft}")

# Soru/cevap alanlarını oluştur
def mtd(d):
    if d is None or (isinstance(d, float) and pd.isna(d)): return ""
    return str(int(d)) if isinstance(d, float) and d == int(d) else str(d).strip()

if ft == "uyds":
    for a in ["modul", "konu", "aciklama", "yapilacak", "yapilan_islem"]:
        veri[f"_{a}"] = veri[eslesme[a]].apply(mtd) if a in eslesme else ""
    veri["_soru_metni"] = (veri["_modul"] + "\n" + veri["_konu"] + "\n" + veri["_aciklama"] + "\n" + veri["_yapilacak"]).str.strip()
    veri["_rag_sorgusu"] = veri["_soru_metni"]
    veri["_cevap_metni"] = veri["_yapilan_islem"].str.strip()
else:
    se = kle({"soru": ["soru"], "cevap": ["cevap"], "kategori": ["kategori"]})
    veri["_modul"] = veri[se["kategori"]].apply(mtd)
    veri["_soru_metni"] = veri[se["soru"]].apply(mtd)
    veri["_rag_sorgusu"] = veri["_soru_metni"]
    veri["_cevap_metni"] = veri[se["cevap"]].apply(mtd)

veri["_cevap_bos"] = veri["_cevap_metni"].str.strip() == ""
print(f"📊 {len(veri)} kayıt, {veri['_cevap_bos'].sum()} boş cevap")

# RAG retrieval
print(f"\n🔎 RAG retrieval (top-{RAG_TOP_K} -> {RAG_SON_K}, min_skor={RAG_MIN_SKOR})...")
sorgular = veri["_rag_sorgusu"].fillna("").tolist()
soru_vektorler = soru_embedding_olustur(sorgular, EMBEDDING_MODEL, batch_size=SORU_EMB_BATCH, cihaz="cpu")

tum_parcalar_list, tum_skorlar = toplu_kaynak_ara(
    soru_vektorler, kaynak_vektorler, tum_parcalar,
    top_k=RAG_TOP_K, son_k=RAG_SON_K, min_skor=RAG_MIN_SKOR,
)

veri["_kaynak_parcalari"] = tum_parcalar_list
veri["_rag_maks_skor"] = tum_skorlar

r_stats = rag_istatistikleri(tum_parcalar_list, tum_skorlar)
print(f"📊 Kaynaksız: {r_stats['kaynaksiz_sorgu']}/{r_stats['toplam_sorgu']}")
print("✅ Hücre 3 tamam.")

## 🤖 Hücre 4: vLLM + Structured Output + Token Kontrollü Prompt

Qwen3-32B-AWQ'yu yükler, chat template ile token kontrolü yapar.

In [ ]:
# ============================================================
# HÜCRE 4: vLLM + Prompt Builder
# ============================================================
from vllm import LLM, SamplingParams
from vllm.sampling_params import StructuredOutputsParams
from src.prompt_builder import (
    SISTEM_PROMPT, DOGRULAMA_PROMPT_SABLON, CEVAP_SEMASI,
    kaynak_formatla, prompt_token_kisalt, token_say,
    kanit_yetersiz_yanit, cevap_bos_yanit, islem_hatasi_yanit,
)
from src.validator import ham_metinden_json_cikar

gc.collect()
torch.cuda.empty_cache()
print(f"🧹 GPU: {torch.cuda.memory_allocated()/1e9:.1f}/{gpu_bellek:.1f} GB")

# vLLM sampling
structured_params = StructuredOutputsParams(json=CEVAP_SEMASI)
sampling_params = SamplingParams(
    temperature=0.2, top_p=0.9, top_k=20,
    max_tokens=MAX_NEW_TOKENS,
    structured_outputs=structured_params,
)

# Tokenizer (prompt kontrolü için)
try:
    from transformers import AutoTokenizer
    _qwen_tk = AutoTokenizer.from_pretrained(BIRINCIL_MODEL, trust_remote_code=False)
    print(f"🔤 Qwen tokenizer: {BIRINCIL_MODEL}")
except:
    _qwen_tk = None
    print("⚠️ Tokenizer yok, karakter/4 fallback")

# vLLM yükle
def vllm_yukle(model_adi):
    try:
        print(f"🔄 {model_adi} yükleniyor...")
        llm = LLM(model=model_adi, max_model_len=MAX_MODEL_LEN,
                   gpu_memory_utilization=GPU_MEM_UTIL, trust_remote_code=False, dtype="auto")
        print(f"✅ Yüklendi.")
        return llm, model_adi
    except Exception as e:
        print(f"❌ {e}")
        return None, None

vllm_llm, KULLANILAN_MODEL = vllm_yukle(BIRINCIL_MODEL)
if vllm_llm is None:
    vllm_llm, KULLANILAN_MODEL = vllm_yukle(YEDEK_MODEL)
if vllm_llm is None:
    raise RuntimeError("❌ Hiçbir model yüklenemedi!")

KAYNAK_EXCEL_HASHI = dosya_hashi(EXCEL_DOSYASI)
print(f"\n🎯 {KULLANILAN_MODEL} | prompt_limit={PROMPT_TOKEN_LIMIT} | batch={BATCH_BUYUKLUK}")
print("✅ Hücre 4 tamam.")

## 🧪 Hücre 5: Benchmark — İlk 100 Kayıt

Batch inference ile hız testi. Checkpoint varsa atlar.

In [ ]:
# ============================================================
# HÜCRE 5: Benchmark (100 kayıt)
# ============================================================
import json, re
from collections import Counter
from src.checkpoint import benchmark_meta_olustur, benchmark_checkpoint_kontrol, sonuc_ekle

BENCHMARK_KAYIT = min(100, len(veri))
BENCHMARK_CKPT = os.path.join(CKPT_DIZINI, "benchmark_sonuclari.jsonl")
BENCHMARK_META = os.path.join(CKPT_DIZINI, "benchmark_meta.json")

# Checkpoint kontrolü
bm_beklenen = benchmark_meta_olustur(
    KAYNAK_EXCEL_HASHI, dizin_hashi(KAYNAK_DIZINI), KULLANILAN_MODEL,
    EMBEDDING_MODEL, PROMPT_SURUMU, KURAL_SETI_SURUMU, CONFIG_HASH,
)
benchmark_sonuclari, bm_yukle = benchmark_checkpoint_kontrol(BENCHMARK_META, BENCHMARK_CKPT, bm_beklenen)

gecen_sure = 0

if not bm_yukle:
    print(f"🧪 Benchmark: {BENCHMARK_KAYIT} kayıt (batch={BATCH_BUYUKLUK})...")
    benchmark_sonuclari = []
    t0 = time.time()

    for batch_basla in range(0, BENCHMARK_KAYIT, BATCH_BUYUKLUK):
        batch_bitir = min(batch_basla + BATCH_BUYUKLUK, BENCHMARK_KAYIT)
        promptlar = []
        kayit_bilgileri = []

        for i in range(batch_basla, batch_bitir):
            idx = veri.index[i]
            if veri.at[idx, "_cevap_bos"]:
                kayit_bilgileri.append((idx, i, True, set(), 0.0))
                continue
            parcalar = veri.at[idx, "_kaynak_parcalari"] or []
            skor = veri.at[idx, "_rag_maks_skor"] or 0.0
            if not parcalar:
                kayit_bilgileri.append((idx, i, True, set(), skor))
                continue

            soru = str(veri.at[idx, "_soru_metni"])
            cevap = str(veri.at[idx, "_cevap_metni"])
            s_k, c_k, k_m, kalan = prompt_token_kisalt(soru, cevap, parcalar, _qwen_tk, PROMPT_TOKEN_LIMIT)
            sunulan = {p["chunk_id"] for p in kalan}
            prompt = DOGRULAMA_PROMPT_SABLON.format(soru=s_k, cevap=c_k, kaynaklar=k_m)
            promptlar.append(prompt)
            kayit_bilgileri.append((idx, i, False, sunulan, skor))

        # Batch çağrı
        if promptlar:
            mesaj_batch = [[{"role":"system","content":SISTEM_PROMPT}, {"role":"user","content":p}] for p in promptlar]
            try:
                ciktilar = vllm_llm.chat(messages=mesaj_batch, sampling_params=sampling_params,
                                        use_tqdm=False, chat_template_kwargs={"enable_thinking":False})
            except Exception as e:
                ciktilar = []

        # Sonuçları işle
        cikti_idx = 0
        dokuman_haritasi = {}
        for idx, i, bos, sunulan, skor in kayit_bilgileri:
            parcalar = veri.at[idx, "_kaynak_parcalari"] or []
            if bos:
                if veri.at[idx, "_cevap_bos"]:
                    sonuc = cevap_bos_yanit()
                else:
                    sonuc = kanit_yetersiz_yanit()
            else:
                cikti = ciktilar[cikti_idx] if cikti_idx < len(ciktilar) else None
                cikti_idx += 1
                if cikti is None or not hasattr(cikti, 'outputs') or not cikti.outputs:
                    sonuc = islem_hatasi_yanit("NoneType")
                else:
                    sonuc = ham_metinden_json_cikar(cikti.outputs[0].text.strip(), sunulan)

            dokumanlar = sorted(set(p["dokuman"] for p in parcalar)) if parcalar else []
            sonuc["_veri_indeksi"] = idx
            sonuc["_kayit_no"] = i
            sonuc["_kaynak_dokumanlari"] = "; ".join(dokumanlar)
            sonuc["_rag_maks_skor"] = skor
            benchmark_sonuclari.append(sonuc)

        gecen = time.time() - t0
        print(f"   {batch_bitir}/{BENCHMARK_KAYIT} | {gecen:.0f}s | ~{(gecen/batch_bitir*len(veri))/60:.0f}dk toplam")

    gecen_sure = time.time() - t0
    with open(BENCHMARK_CKPT, "w", encoding="utf-8") as f:
        for s in benchmark_sonuclari:
            f.write(json.dumps(s, ensure_ascii=False) + "\n")
    with open(BENCHMARK_META, "w") as f:
        json.dump(bm_beklenen, f, ensure_ascii=False, indent=2)

# Özet
kararlar = [s.get("karar","?") for s in benchmark_sonuclari]
guvenler = [s.get("guven",0) for s in benchmark_sonuclari if not s.get("_islem_hatasi")]
hatalar = sum(1 for s in benchmark_sonuclari if s.get("_islem_hatasi"))

print(f"\n📊 BENCHMARK: {len(benchmark_sonuclari)} kayıt")
print(f"   Karar: {dict(Counter(kararlar))}")
if guvenler:
    print(f"   Güven ort: {np.mean(guvenler):.3f}")
if hatalar:
    print(f"   🔴 İşlem hatası: {hatalar}")

if gecen_sure > 0:
    print(f"   ⏱️ {gecen_sure/BENCHMARK_KAYIT:.1f}s/kayıt → ~{gecen_sure/BENCHMARK_KAYIT*len(veri)/60:.0f}dk toplam")

print("✅ Hücre 5 tamam.")

## ⚡ Hücre 6: Toplu Doğrulama + Checkpoint

Tüm kayıtları batch'ler halinde doğrular, sonuc_haritasi ile checkpoint yönetir.

In [ ]:
# ============================================================
# HÜCRE 6: Toplu Doğrulama + Checkpoint
# ============================================================
from src.checkpoint import checkpoint_yukle, sonuc_ekle, eksik_kayitlari_bul, butunluk_kontrolu

SONUC_CKPT = os.path.join(CKPT_DIZINI, "dogrulama_sonuclari.jsonl")
META_CKPT = os.path.join(CKPT_DIZINI, "meta.json")

# sonuc_haritasi
sonuc_haritasi = {}

# Benchmark sonuçlarını ekle
if os.path.exists(BENCHMARK_CKPT):
    with open(BENCHMARK_CKPT, "r", encoding="utf-8") as f:
        for satir in f:
            if satir.strip():
                try:
                    s = json.loads(satir)
                    sonuc_haritasi[int(s["_kayit_no"])] = s
                except:
                    pass
    print(f"📋 {len([k for k in sonuc_haritasi if k < BENCHMARK_KAYIT])} benchmark sonucu yüklendi.")

# Ana checkpoint
beklenen_meta = {
    "excel_hash": KAYNAK_EXCEL_HASHI, "kaynak_dizin_hash": dizin_hashi(KAYNAK_DIZINI),
    "kullanilan_model": KULLANILAN_MODEL, "embedding_model": EMBEDDING_MODEL,
    "prompt_surumu": PROMPT_SURUMU, "kural_seti_surumu": KURAL_SETI_SURUMU,
    "config_hash": CONFIG_HASH,
}
ckpt_harita, ckpt_gecerli = checkpoint_yukle(SONUC_CKPT, META_CKPT, beklenen_meta)
if ckpt_gecerli:
    for k, v in ckpt_harita.items():
        sonuc_haritasi[k] = v

# Eksikleri belirle
tamamlanan, hatali, eksik_nolar = eksik_kayitlari_bul(sonuc_haritasi, len(veri))

print(f"\n⚡ Toplu doğrulama: {len(veri)} kayıt")
print(f"   Tamamlanan: {len(tamamlanan)} | Hatalı: {len(hatali)} | Eksik: {len(eksik_nolar)}")

if not eksik_nolar:
    print("✅ Tüm kayıtlar zaten tamamlanmış.")
else:
    t0 = time.time()
    islenen = 0
    ckpt_dosyasi = open(SONUC_CKPT, "a", encoding="utf-8")

    try:
        for batch_basla in range(0, len(eksik_nolar), BATCH_BUYUKLUK):
            batch_nolar = eksik_nolar[batch_basla:batch_basla + BATCH_BUYUKLUK]
            if not batch_nolar:
                continue
            promptlar = []
            kayit_bilgileri = []

            for i in batch_nolar:
                idx = veri.index[i]
                if veri.at[idx, "_cevap_bos"]:
                    kayit_bilgileri.append((idx, i, True, set(), 0.0))
                    continue
                parcalar = veri.at[idx, "_kaynak_parcalari"] or []
                skor = veri.at[idx, "_rag_maks_skor"] or 0.0
                if not parcalar:
                    kayit_bilgileri.append((idx, i, True, set(), skor))
                    continue

                soru = str(veri.at[idx, "_soru_metni"])
                cevap = str(veri.at[idx, "_cevap_metni"])
                s_k, c_k, k_m, kalan = prompt_token_kisalt(soru, cevap, parcalar, _qwen_tk, PROMPT_TOKEN_LIMIT)
                sunulan = {p["chunk_id"] for p in kalan}
                prompt = DOGRULAMA_PROMPT_SABLON.format(soru=s_k, cevap=c_k, kaynaklar=k_m)
                promptlar.append(prompt)
                kayit_bilgileri.append((idx, i, False, sunulan, skor))

            if promptlar:
                mesaj_batch = [[{"role":"system","content":SISTEM_PROMPT}, {"role":"user","content":p}] for p in promptlar]
                try:
                    ciktilar = vllm_llm.chat(messages=mesaj_batch, sampling_params=sampling_params,
                                            use_tqdm=False, chat_template_kwargs={"enable_thinking":False})
                except:
                    ciktilar = []

            cikti_idx = 0
            for idx, i, bos, sunulan, skor in kayit_bilgileri:
                parcalar = veri.at[idx, "_kaynak_parcalari"] or []
                if bos:
                    sonuc = cevap_bos_yanit() if veri.at[idx, "_cevap_bos"] else kanit_yetersiz_yanit()
                else:
                    cikti = ciktilar[cikti_idx] if cikti_idx < len(ciktilar) else None
                    cikti_idx += 1
                    if cikti is None or not hasattr(cikti, 'outputs') or not cikti.outputs:
                        sonuc = islem_hatasi_yanit("NoneType")
                    else:
                        sonuc = ham_metinden_json_cikar(cikti.outputs[0].text.strip(), sunulan)

                dokumanlar = sorted(set(p["dokuman"] for p in parcalar)) if parcalar else []
                sonuc["_veri_indeksi"] = idx
                sonuc["_kayit_no"] = i
                sonuc["_kaynak_dokumanlari"] = "; ".join(dokumanlar)
                sonuc["_rag_maks_skor"] = skor
                sonuc_haritasi[i] = sonuc
                ckpt_dosyasi.write(json.dumps(sonuc, ensure_ascii=False) + "\n")
                islenen += 1

            gecen = time.time() - t0
            print(f"   {len(sonuc_haritasi)}/{len(veri)} | {gecen/60:.1f}dk")
    finally:
        ckpt_dosyasi.flush()
        ckpt_dosyasi.close()

    toplam_sure = time.time() - t0
    print(f"\n⏱️ Toplu doğrulama: {toplam_sure/60:.1f} dk")

# Final bütünlük
tum_sonuclar = []
eksik_final = []
hata_final = []
for i in range(len(veri)):
    if i in sonuc_haritasi:
        s = sonuc_haritasi[i]
        if s.get("_islem_hatasi"):
            hata_final.append(i)
        tum_sonuclar.append(s)
    else:
        eksik_final.append(i)

if eksik_final:
    raise RuntimeError(f"❌ {len(eksik_final)} kayıt doğrulanmadı: {eksik_final[:10]}")
print(f"✅ Tüm {len(veri)} kayıt işlendi.")
print("✅ Hücre 6 tamam.")

## 📊 Hücre 7: Konsolidasyon + Çıktılar + Grafikler

`src/reporting` modülü tüm Excel/JSON/grafik çıktılarını üretir.

In [ ]:
# ============================================================
# HÜCRE 7: Çıktılar ve Raporlar
# ============================================================
from src.reporting import (
    tum_sonuclari_excel_kaydet, kaynak_kapsama_raporu,
    ozet_json_uret, grafik_uret, benchmark_ozeti_uret,
)

cikti_df, dogrulanan, supheli, hata_sayisi = tum_sonuclari_excel_kaydet(
    veri, tum_sonuclar, CIKTI_DIZINI, guven_esik=GUVEN_ESIK,
)

# Karar dağılımı
print(f"\n📊 Karar dağılımı ({len(cikti_df)} kayıt):")
for k, v in cikti_df["_karar"].value_counts().items():
    print(f"   {k:20s}: {v:5d} (%{100*v/len(cikti_df):.1f})")

print(f"\n   ✅ Doğrulanan: {len(dogrulanan)}")
print(f"   ⚠️ Şüpheli: {len(supheli)}")
if hata_sayisi:
    print(f"   🔴 İşlem hatası: {hata_sayisi}")

# Kapsama raporu
kapsama = kaynak_kapsama_raporu(cikti_df, tum_parcalar, CIKTI_DIZINI)

# Özet JSON
ozet = ozet_json_uret(cikti_df, dogrulanan, supheli, hata_sayisi, CIKTI_DIZINI, ek_bilgiler={
    "surum": "4.1.0",
    "model": KULLANILAN_MODEL,
    "embedding_model": EMBEDDING_MODEL,
    "config": {
        "chunk_token": CHUNK_BOYUT_TOKEN, "chunk_overlap": CHUNK_OVERLAP_TOKEN,
        "rag_top_k": RAG_TOP_K, "rag_son_k": RAG_SON_K,
        "max_model_len": MAX_MODEL_LEN, "prompt_token_limit": PROMPT_TOKEN_LIMIT,
        "batch": BATCH_BUYUKLUK, "guven_esik": GUVEN_ESIK,
    },
    "excel": os.path.basename(EXCEL_DOSYASI),
    "kaynak_kapsama": kapsama,
})

# Benchmark özeti
bm = benchmark_ozeti_uret(benchmark_sonuclari, CIKTI_DIZINI, gecen_sure, len(veri))

# Grafik
grafik = grafik_uret(cikti_df, CIKTI_DIZINI, guven_esik=GUVEN_ESIK)

# Çıktı listesi
print(f"\n📁 Çıktılar ({CIKTI_DIZINI}):")
for df in sorted(os.listdir(CIKTI_DIZINI)):
    boyut = os.path.getsize(os.path.join(CIKTI_DIZINI, df)) / 1024
    print(f"   - {df} ({boyut:.1f} KB)")

print(f"\n🎉 Pipeline v4.1.0 tamamlandı!")
print("✅ Hücre 7 tamam.")

## 🧹 Hücre 8: Temizlik

GPU belleğini boşaltır, isteğe bağlı.

In [ ]:
# ============================================================
# HÜCRE 8: Temizlik
# ============================================================
print("🧹 Temizlik...")
try:
    del vllm_llm
except:
    pass
gc.collect()
torch.cuda.empty_cache()
print(f"   GPU: {torch.cuda.memory_allocated()/1e9:.1f}/{gpu_bellek:.1f} GB")
print("✅ Temizlik tamam.")